# Pydantic Basic Usage

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

In [7]:
from pydantic import BaseModel, Field, ValidationError
from rich import print as rprint
from typing import Optional

class Person(BaseModel):
    """Personal Information"""
    name: str = Field(description="Personal Name")
    age: Optional[int] = Field(description="Personal Age")
    occupation: str = Field(description="Personal Occupation")

structured_model = model.with_structured_output(Person)

result = structured_model.invoke("Mark is an experienced engineer")

rprint(result)
print(type(result))

Person(name='Mark', age=None, occupation='engineer')

<class '__main__.Person'>


In [4]:
from typing import List
class PersonList(BaseModel):
    people: List[Person]

structured_model = model.with_structured_output(PersonList)
result = structured_model.invoke("张三 30岁，李四 40岁")
print(result)

people=[Person(name='张三', age=30, occupation=''), Person(name='李四', age=40, occupation='')]


In [5]:
class User(BaseModel):
    name: str = Field(description="User's Name", min_length=2, max_length=50)
    age: int = Field(description="User's Age", le = 150)
    email: str = Field(description="User's Email")

user1 = User(name="Tom", age=20, email="tom@gmail.com")
print(user1)


name='Tom' age=20 email='tom@gmail.com'


In [8]:
try:
    user2 = User(name="Tom", age=200, email="tom@gmail.com")
    print("[OK]")
except ValidationError as e:
    print(f"[FAIL] {e}")


[FAIL] 1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
